# TabSyn × RF — Rayong minority crop augmentation

Apply [TabSyn](https://github.com/amazon-science/tabsyn) (latent diffusion over VAE) to Rayong crop data. Generate synthetic samples for minority classes, then compare RF cascade trained on real-only vs real+synth.

Run AFTER `rayong_rf_cascade.ipynb` (needs `baseline_dataset.parquet`).

Pipeline:
1. Build minority-only CSV from baseline parquet
2. `process_dataset.py` (TabSyn schema)
3. Train VAE → encode latents
4. Train latent diffusion (logs to `loss_history.csv`)
5. Sample synthetic → `synthetic/<DATASET>/tabsyn.csv`
6. KS / class balance / correlation eval
7. RF cascade train: real-only vs real+synth (same 2-stage architecture as `rayong_rf_cascade.ipynb`)
8. Per-class F1 + 2×2 CM grid

## 1. Config

In [ ]:
import os, sys, subprocess, json, time, joblib, gc
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt

HERE     = Path.cwd()
PARQUET  = Path(r'D:\work\GIS\rf\preprocess\baseline_dataset.parquet')
DATA_DIR = HERE / 'data'
INFO_DIR = DATA_DIR / 'Info'
INFO_DIR.mkdir(parents=True, exist_ok=True)

INDEX_NAMES  = ['ndvi', 'evi', 'mndwi', 'mtci', 'swir']
MONTHS       = [10, 11, 12]
FEATURE_COLS = [f'{idx} {m}' for m in MONTHS for idx in INDEX_NAMES]
TARGET_COL   = 'label'
KEEP         = FEATURE_COLS + [TARGET_COL]
LABEL_IDX    = len(FEATURE_COLS)

LABEL_NAMES = {
    2101: 'Paddy', 2204: 'Cassava', 2205: 'Pineapple', 2302: 'Rubber', 2303: 'OilPalm',
    2403: 'Durian', 2404: 'Rambutan', 2405: 'Coconut', 2407: 'Mango', 2413: 'Longan',
    2416: 'Jackfruit', 2419: 'Mangosteen', 2420: 'Langsat', 4201: 'Reservoir', 9999: 'OTHER',
}
def lname(c): return f'{int(c)} {LABEL_NAMES.get(int(c), "?")}'

OTHERS_LABEL  = 9999
MIN_THRESHOLD = 100_000
PER_CLASS_CAP = 400_000
SEED          = 42
DATASET       = 'crop_baseline_min'

print('parquet exists:', PARQUET.exists(), PARQUET)

## 2. Build minority-only CSV

In [ ]:
def emit(name, df):
    out = DATA_DIR / name
    out.mkdir(parents=True, exist_ok=True)
    csv = out / f'{name}.csv'
    df[KEEP].to_csv(csv, header=False, index=False)
    print(f'  {csv}  rows={len(df):,}  {csv.stat().st_size/1e6:.1f} MB')

df = pd.read_parquet(PARQUET, columns=KEEP)
df[TARGET_COL] = df[TARGET_COL].astype(np.int32)
counts = df[TARGET_COL].value_counts().sort_index()
for c, n in counts.items():
    print(f'  {lname(c):<24s}: {n:>10,}')

minority = [int(c) for c, n in counts.items() if c != OTHERS_LABEL and n < MIN_THRESHOLD]
print(f'\nminority: {[lname(c) for c in minority]}')
df_min = df[df[TARGET_COL].isin(minority)].sample(frac=1, random_state=SEED).reset_index(drop=True)
emit(DATASET, df_min)
del df, df_min

## 3. Write info.json (TabSyn metadata)

In [ ]:
INFO = {
    'name': DATASET, 'task_type': 'multiclass', 'header': None,
    'column_names': KEEP,
    'num_col_idx': list(range(LABEL_IDX)), 'cat_col_idx': [], 'target_col_idx': [LABEL_IDX],
    'file_type': 'csv',
    'data_path': f'data/{DATASET}/{DATASET}.csv', 'test_path': None,
}
(INFO_DIR / f'{DATASET}.json').write_text(json.dumps(INFO, indent=4))
print(f'wrote {INFO_DIR / f"{DATASET}.json"}')

## 4. process_dataset.py + train VAE + train diffusion

In [ ]:
def run(cmd):
    print('$', ' '.join(cmd))
    env = dict(os.environ); env.setdefault('PYTHONUNBUFFERED', '1'); env.setdefault('PYTHONIOENCODING', 'utf-8')
    p = subprocess.Popen(cmd, cwd=str(HERE), env=env, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1, encoding='utf-8', errors='replace')
    for line in p.stdout: print(line, end='', flush=True)
    p.wait()
    if p.returncode: raise RuntimeError(f'cmd failed: {cmd}')

run([sys.executable, 'process_dataset.py', '--dataname', DATASET])
run([sys.executable, 'main.py', '--dataname', DATASET, '--method', 'vae',    '--mode', 'train'])
run([sys.executable, 'main.py', '--dataname', DATASET, '--method', 'tabsyn', '--mode', 'train'])

## 5. Diffusion training loss curve

In [ ]:
LOSS_CSV = HERE / 'tabsyn' / 'ckpt' / DATASET / 'loss_history.csv'
if LOSS_CSV.exists():
    loss_df = pd.read_csv(LOSS_CSV)
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
    axes[0].plot(loss_df['epoch'], loss_df['train_loss'], label='train', color='#1d4ed8', linewidth=1.3)
    axes[0].plot(loss_df['epoch'], loss_df['best_loss'],  label='best', color='#15803d', linestyle='--', linewidth=1.3)
    if len(loss_df) >= 10:
        axes[0].plot(loss_df['epoch'], loss_df['train_loss'].rolling(10, min_periods=1).mean(),
                     label='roll mean (10)', color='#dc2626', linewidth=1.5)
    axes[0].set_xlabel('epoch'); axes[0].set_ylabel('loss')
    axes[0].set_title('TabSyn diffusion loss'); axes[0].grid(alpha=0.3); axes[0].legend()
    axes[1].plot(loss_df['epoch'], loss_df['lr'], color='#a16207', linewidth=1.3)
    axes[1].set_xlabel('epoch'); axes[1].set_ylabel('lr'); axes[1].set_yscale('log')
    axes[1].set_title('LR schedule'); axes[1].grid(alpha=0.3, which='both')
    plt.tight_layout(); plt.show()
    print(f'best loss {loss_df["best_loss"].min():.4f} at epoch {int(loss_df.loc[loss_df["best_loss"].idxmin(), "epoch"])}')
else:
    print(f'  [skip] {LOSS_CSV} not found')

## 6. Sample synthetic

In [ ]:
SYN_DIR  = HERE / 'synthetic' / DATASET; SYN_DIR.mkdir(parents=True, exist_ok=True)
SYN_PATH = SYN_DIR / 'tabsyn.csv'
run([sys.executable, 'main.py', '--dataname', DATASET, '--method', 'tabsyn', '--mode', 'sample',
     '--save_path', str(SYN_PATH)])
print('synth file:', SYN_PATH, 'size:', SYN_PATH.stat().st_size/1e6, 'MB')

## 7. Quality eval — class balance + KS + density overlay

In [ ]:
real = pd.read_csv(DATA_DIR / DATASET / 'train.csv')
syn  = pd.read_csv(SYN_PATH)
if list(syn.columns) != list(real.columns): syn.columns = real.columns
valid_classes = sorted(real[TARGET_COL].astype(int).unique())
syn[TARGET_COL] = syn[TARGET_COL].astype(float).round().astype(int)
syn[TARGET_COL] = syn[TARGET_COL].clip(min(valid_classes), max(valid_classes))
_arr = np.array(valid_classes)
_snap = lambda v: int(_arr[np.argmin(np.abs(_arr - v))])
syn[TARGET_COL] = syn[TARGET_COL].apply(_snap)

balance = pd.DataFrame({
    'real': real[TARGET_COL].value_counts().sort_index(),
    'syn':  syn [TARGET_COL].value_counts().sort_index(),
}).fillna(0).astype(int)
balance.index = [lname(c) for c in balance.index]
print(balance.to_string())

from scipy.stats import ks_2samp
ks_rows = [{'feat': c, 'ks_stat': ks_2samp(real[c], syn[c])[0]} for c in FEATURE_COLS]
print('\nKS test (lower = closer):')
print(pd.DataFrame(ks_rows).round(4).to_string(index=False))

fig, axs = plt.subplots(2, 3, figsize=(14, 6))
for ax, col in zip(axs.flat, FEATURE_COLS[:6]):
    bins = np.linspace(min(real[col].min(), syn[col].min()),
                       max(real[col].max(), syn[col].max()), 60)
    ax.hist(real[col], bins=bins, alpha=0.5, label='real', density=True)
    ax.hist(syn [col], bins=bins, alpha=0.5, label='syn',  density=True)
    ax.set_title(col); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 8. Train RF cascade — real-only vs real+synth

Same 2-stage architecture as `rayong_rf_cascade.ipynb`: stage-1 base RF (15 cls) + stage-2 tropical-orchard specialist (8 cls). Synth lifts each minority to `SYN_TARGET`.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, cohen_kappa_score, classification_report, ConfusionMatrixDisplay

DEFAULT_CAP = 400_000
SYN_TARGET  = 200_000
RF_PARAMS = dict(n_estimators=300, max_depth=36, min_samples_split=5, min_samples_leaf=1,
                 max_features='log2', class_weight='balanced_subsample', bootstrap=True,
                 n_jobs=-1, random_state=SEED)
RF_PARAMS_ORCH = dict(RF_PARAMS, class_weight='balanced')
TROPICAL_ORCHARDS = np.array([2403, 2404, 2405, 2407, 2413, 2416, 2419, 2420], dtype=np.int32)

df_real = pd.read_parquet(PARQUET, columns=KEEP + ['parcel_id'])
df_real[TARGET_COL]  = df_real[TARGET_COL].astype(np.int32)
df_real['parcel_id'] = df_real['parcel_id'].astype(np.int64)

rng = np.random.default_rng(SEED)
real_parts = []
for c in np.unique(df_real[TARGET_COL]):
    idx = np.flatnonzero(df_real[TARGET_COL].to_numpy() == c)
    if len(idx) > DEFAULT_CAP:
        idx = rng.choice(idx, DEFAULT_CAP, replace=False)
    real_parts.append(idx)
df_real = df_real.iloc[np.concatenate(real_parts)].reset_index(drop=True)

y_real = df_real[TARGET_COL].to_numpy()
X_real = df_real[FEATURE_COLS].to_numpy(dtype=np.float32)
pid_r  = df_real['parcel_id'].to_numpy(dtype=np.int64)
X_tv, X_te, y_tv, y_te, pid_tv, pid_te = train_test_split(
    X_real, y_real, pid_r, test_size=0.10, stratify=y_real, random_state=SEED)
X_tr_real, X_va, y_tr_real, y_va, pid_tr_real, pid_va = train_test_split(
    X_tv, y_tv, pid_tv, test_size=1/9, stratify=y_tv, random_state=SEED)
del df_real, X_real, y_real, pid_r, X_tv, y_tv, pid_tv; gc.collect()

syn_full = pd.read_csv(SYN_PATH)
if list(syn_full.columns) != KEEP: syn_full.columns = KEEP
syn_full[TARGET_COL] = syn_full[TARGET_COL].astype(float).round().astype(int)
syn_full[TARGET_COL] = syn_full[TARGET_COL].clip(min(valid_classes), max(valid_classes)).apply(_snap)

minority = [int(c) for c, n in pd.Series(y_tr_real).value_counts().items() if n < SYN_TARGET]
aug_X_parts, aug_y_parts, aug_pid_parts = [X_tr_real], [y_tr_real], [pid_tr_real]
for c in minority:
    cur_n = int((y_tr_real == c).sum())
    need  = max(0, SYN_TARGET - cur_n)
    syn_c = syn_full[syn_full[TARGET_COL] == c]
    if not len(syn_c) or need == 0: continue
    pick = syn_c.sample(min(need, len(syn_c)), replace=(len(syn_c) < need), random_state=SEED + c)
    aug_X_parts.append(pick[FEATURE_COLS].to_numpy(dtype=np.float32))
    aug_y_parts.append(pick[TARGET_COL].to_numpy(dtype=np.int32))
    aug_pid_parts.append(-np.arange(1, len(pick) + 1, dtype=np.int64) - 1_000_000_000 * c)
X_tr_aug = np.concatenate(aug_X_parts)
y_tr_aug = np.concatenate(aug_y_parts).astype(np.int32)
pid_tr_aug = np.concatenate(aug_pid_parts)
print(f'train_real: {X_tr_real.shape}  train_aug: {X_tr_aug.shape}')

def fit_cascade(X_tr, y_tr, name):
    t0 = time.time()
    clf_s1 = RandomForestClassifier(**RF_PARAMS).fit(X_tr, y_tr)
    mask = np.isin(y_tr, TROPICAL_ORCHARDS)
    clf_s2 = RandomForestClassifier(**RF_PARAMS_ORCH).fit(X_tr[mask], y_tr[mask]) if mask.sum() >= 100 else None
    print(f'{name}: fit {time.time()-t0:.1f}s  s1_classes={clf_s1.classes_.tolist()}')
    return clf_s1, clf_s2

def cascade(clf_s1, clf_s2):
    def _p(X_):
        pred = clf_s1.predict(X_).astype(np.int32)
        if clf_s2 is not None:
            m = np.isin(pred, TROPICAL_ORCHARDS)
            if m.any(): pred[m] = clf_s2.predict(X_[m]).astype(np.int32)
        return pred
    return _p

clf_real_s1, clf_real_s2 = fit_cascade(X_tr_real, y_tr_real, 'rf_real')
clf_aug_s1,  clf_aug_s2  = fit_cascade(X_tr_aug,  y_tr_aug,  'rf_aug')
predict_real = cascade(clf_real_s1, clf_real_s2)
predict_aug  = cascade(clf_aug_s1,  clf_aug_s2)

## 9. Eval — pixel + parcel-vote, real-only vs augmented

In [ ]:
CLF_CLASSES = np.asarray(sorted(np.unique(y_te)), dtype=np.int32)
display_labels = [lname(c) for c in CLF_CLASSES]

def evalp(name, fn):
    yp = fn(X_te)
    f1  = f1_score(y_te, yp, average=None, labels=CLF_CLASSES.tolist(), zero_division=0)
    print(f'\n[{name}] acc={accuracy_score(y_te, yp):.4f}  '
          f'weighted_F1={f1_score(y_te, yp, average="weighted", zero_division=0):.4f}  '
          f'macro_F1={f1_score(y_te, yp, average="macro", zero_division=0):.4f}  '
          f'kappa={cohen_kappa_score(y_te, yp):.4f}')
    print(classification_report(y_te, yp, labels=CLF_CLASSES.tolist(),
                                target_names=display_labels, digits=4, zero_division=0))
    return yp, pd.Series(f1, index=CLF_CLASSES, name=name)

def parcel_vote(name, yp):
    df = pd.DataFrame({'pid': pid_te, 'y': y_te.astype(np.int32), 'yp': yp.astype(np.int32)})
    grp = df.groupby('pid', sort=False)
    py  = grp['y'].first().to_numpy()
    ypp = grp['yp'].agg(lambda s: int(s.mode().iloc[0])).to_numpy()
    print(f'\n[{name} parcel-vote] n={len(py):,}  '
          f'acc={accuracy_score(py, ypp):.4f}  '
          f'macro_F1={f1_score(py, ypp, average="macro", zero_division=0):.4f}')
    return py, ypp

yp_te_real, f1_real_pix = evalp('real-only cascade', predict_real)
yp_te_aug,  f1_aug_pix  = evalp('augmented cascade', predict_aug)
py_te_real, ypp_te_real = parcel_vote('real-only', yp_te_real)
py_te_aug,  ypp_te_aug  = parcel_vote('augmented', yp_te_aug)

cmp = pd.DataFrame({'real-only': f1_real_pix, 'augmented': f1_aug_pix})
cmp['delta'] = cmp['augmented'] - cmp['real-only']
cmp.index = display_labels
print('\n=== per-class F1 compare ===')
print(cmp.round(4).to_string())

## 10. Per-class F1 bar + 2×2 CM grid

In [ ]:
ART = HERE / 'rf_artifacts'; ART.mkdir(exist_ok=True)

fig, ax = plt.subplots(figsize=(13, 5))
x = np.arange(len(cmp))
ax.bar(x - 0.2, cmp['real-only'], 0.4, label='real-only')
ax.bar(x + 0.2, cmp['augmented'], 0.4, label='augmented (TabSyn)')
ax.set_xticks(x); ax.set_xticklabels(cmp.index, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('F1'); ax.set_title('Per-class F1 — real-only vs TabSyn-augmented')
ax.legend(); plt.tight_layout()
plt.savefig(ART / 'tabsyn_per_class_f1.png', dpi=120, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(22, 18))
panels = [
    (axes[0, 0], 'real-only PIXEL',       y_te,       yp_te_real,  'Blues'),
    (axes[0, 1], 'augmented PIXEL',       y_te,       yp_te_aug,   'Blues'),
    (axes[1, 0], 'real-only PARCEL-VOTE', py_te_real, ypp_te_real, 'Greens'),
    (axes[1, 1], 'augmented PARCEL-VOTE', py_te_aug,  ypp_te_aug,  'Greens'),
]
for ax, title, y_true, y_pred, cmap in panels:
    ConfusionMatrixDisplay.from_predictions(
        y_true, y_pred, labels=CLF_CLASSES.tolist(), display_labels=display_labels,
        normalize='true', cmap=cmap, ax=ax, colorbar=False, xticks_rotation=90, values_format='.2f')
    ax.set_title(f'{title}', fontsize=12)
plt.tight_layout()
plt.savefig(ART / 'tabsyn_cm_combined.png', dpi=120, bbox_inches='tight')
plt.show()

joblib.dump({'clf_s1': clf_real_s1, 'clf_s2': clf_real_s2, 'feature_cols': FEATURE_COLS,
             'label_names': LABEL_NAMES, 'tropical_orchards': TROPICAL_ORCHARDS,
             'rf_params': RF_PARAMS, 'training': 'real_only_cascade'},
            ART / 'rf_real_baseline.joblib', compress=3)
joblib.dump({'clf_s1': clf_aug_s1,  'clf_s2': clf_aug_s2,  'feature_cols': FEATURE_COLS,
             'label_names': LABEL_NAMES, 'tropical_orchards': TROPICAL_ORCHARDS,
             'rf_params': RF_PARAMS, 'training': 'real_plus_tabsyn_synth_cascade',
             'syn_target': SYN_TARGET, 'minority_classes': minority},
            ART / 'rf_aug_baseline.joblib', compress=3)
print(f'saved -> {ART}/')

## 11. Deploy export — TabSyn artifacts for streamlit (`deploy/artifacts/`)

Writes the three files Model Card needs (`tabsyn_metrics.json`, `tabsyn_confusion.npy`,
`tabsyn_feature_importance.parquet`) plus `synth_tabsyn.parquet` + `corr_syn.npy`
for the Synth Lab page. Heavy joblibs go to a side dir (web does not run live
inference). Streamlit's `_norm()` auto-renames `mndwi *` → `ndwi *`, so
no schema rewrite is needed here.

In [ ]:
from sklearn.metrics import confusion_matrix

DEMO_DIR = Path(r'D:\Github\synth-crop-web\deploy\artifacts')
DEMO_DIR.mkdir(parents=True, exist_ok=True)

JOBLIB_DIR = Path(r'D:\work\GIS\rf\rf_artifacts_tabsyn')
JOBLIB_DIR.mkdir(parents=True, exist_ok=True)

# ---- 1. synth_tabsyn.parquet (TabSyn output) ----
syn_out = syn_full[FEATURE_COLS + [TARGET_COL]].copy()
syn_out.to_parquet(DEMO_DIR / 'synth_tabsyn.parquet', compression='zstd', index=False)
print(f'wrote synth_tabsyn.parquet  ({len(syn_out):,} rows)')

# ---- 2. corr_syn.npy (feature x feature correlation on synth) ----
sample = syn_out.sample(min(200_000, len(syn_out)), random_state=SEED)
corr_syn = sample[FEATURE_COLS].corr().values.astype(np.float32)
np.save(DEMO_DIR / 'corr_syn.npy', corr_syn)
print(f'wrote corr_syn.npy  shape={corr_syn.shape}')

# ---- 3. tabsyn_metrics.json (Model Card schema: pixel_test/parcel_test + f1_per_class) ----
def _pix(yp):
    return {
        'acc':         float(accuracy_score(y_te, yp)),
        'kappa':       float(cohen_kappa_score(y_te, yp)),
        'f1_weighted': float(f1_score(y_te, yp, average='weighted', zero_division=0)),
        'f1_macro':    float(f1_score(y_te, yp, average='macro',    zero_division=0)),
        'f1_per_class': [float(v) for v in f1_score(y_te, yp, average=None,
                                                    labels=CLF_CLASSES.tolist(),
                                                    zero_division=0)],
    }

def _par(py, ypp):
    return {
        'n_parcels':   int(len(py)),
        'acc':         float(accuracy_score(py, ypp)),
        'kappa':       float(cohen_kappa_score(py, ypp)),
        'f1_weighted': float(f1_score(py, ypp, average='weighted', zero_division=0)),
        'f1_macro':    float(f1_score(py, ypp, average='macro',    zero_division=0)),
    }

tabsyn_metrics = {
    'model_label':          'Random Forest 2-stage cascade + TabSyn synthetic minority augmentation',
    'cascade_stages':       'stage-1 base RF (15 cls) + stage-2 tropical-orchard specialist (8 cls)',
    'rf_params':            RF_PARAMS,
    'rf_params_s2':         RF_PARAMS_ORCH,
    'tropical_orchards':    TROPICAL_ORCHARDS.tolist(),
    'minority_classes':     minority,
    'syn_target_per_class': SYN_TARGET,
    'feature_cols':         FEATURE_COLS,
    'classes':              CLF_CLASSES.tolist(),
    'label_names':          {str(int(c)): LABEL_NAMES.get(int(c), '?') for c in CLF_CLASSES},
    'pixel_test':           _pix(yp_te_aug),
    'parcel_test':          _par(py_te_aug, ypp_te_aug),
    'real_only_cascade':    {'pixel_test': _pix(yp_te_real),
                             'parcel_test': _par(py_te_real, ypp_te_real)},
}
(DEMO_DIR / 'tabsyn_metrics.json').write_text(json.dumps(tabsyn_metrics, indent=2, ensure_ascii=False))
print('wrote tabsyn_metrics.json')

# ---- 4. tabsyn_confusion.npy (raw counts, augmented pixel cascade) ----
np.save(DEMO_DIR / 'tabsyn_confusion.npy',
        confusion_matrix(y_te, yp_te_aug, labels=CLF_CLASSES.tolist()))
print('wrote tabsyn_confusion.npy')

# ---- 5. tabsyn_feature_importance.parquet (augmented stage-1) ----
pd.DataFrame({'feature': FEATURE_COLS, 'importance': clf_aug_s1.feature_importances_}) \
  .sort_values('importance', ascending=False) \
  .to_parquet(DEMO_DIR / 'tabsyn_feature_importance.parquet', index=False)
print('wrote tabsyn_feature_importance.parquet')

# ---- 6. joblibs (side dir, NOT deploy) ----
joblib.dump({
    'clf_s1': clf_aug_s1, 'clf_s2': clf_aug_s2,
    'feature_cols': FEATURE_COLS, 'label_names': LABEL_NAMES,
    'tropical_orchards': TROPICAL_ORCHARDS,
    'rf_params': RF_PARAMS, 'rf_params_s2': RF_PARAMS_ORCH,
    'training': 'real_plus_tabsyn_synth_minorities_cascade',
    'syn_target': SYN_TARGET, 'minority_classes': minority,
}, JOBLIB_DIR / 'rf_tabsyn_baseline.joblib', compress=3)
print(f'wrote {JOBLIB_DIR}/rf_tabsyn_baseline.joblib')

print(f'\n=== deploy artifacts in {DEMO_DIR} ===')
for fname in ['synth_tabsyn.parquet', 'corr_syn.npy', 'tabsyn_metrics.json',
              'tabsyn_confusion.npy', 'tabsyn_feature_importance.parquet']:
    p = DEMO_DIR / fname
    if p.exists():
        print(f'  {fname:<36s} {p.stat().st_size/1e6:>7.2f} MB')